In [1]:
import asyncio, os
from decimal import Decimal
from datetime import datetime, timedelta, timezone

from typed_dydx import Dydx
from typed_dydx.indexer.schemas import (
  OrderBook,
  Order as IndexerOrder,
  OrderStatus,
  PerpetualMarket,
)
from typed_dydx.indexer.streams.orders import OrderbookMessageContents
from dotenv import load_dotenv

from tribulnation.sdk.core import ApiError, ValidationError
from tribulnation.sdk.market import (
  Book,
  FundingPayment,
  FundingRate,
  NextFunding,
  Order,
  OrderResponse,
  OrderState,
  PerpCollateral,
  PerpPosition,
  PerpStats,
  Rules,
  Ticker,
  Trade,
)

load_dotenv()

MARKETS = ['BTC-USD', 'ETH-USD', 'SOL-USD']

# The only dYdX credentials available here are testnet-only (`DYDX_TESTNET_ADDRESS` /
# `DYDX_TESTNET_MNEMONIC`), so every call below -- public market data included -- goes
# through dYdX's testnet indexer/chain/node endpoints.
client = Dydx.testnet(os.environ['DYDX_TESTNET_MNEMONIC'], indexer={'validate': True})
await client.__aenter__()
address = os.environ['DYDX_TESTNET_ADDRESS']

> This notebook hand-maps `typed_dydx`'s raw responses onto `tribulnation.sdk.market`
> types directly -- it does not import or call `tribulnation.dydx` at all. Every book,
> market, position, and collateral figure below reflects dYdX's **testnet**
> (`DYDX_TESTNET_ADDRESS` / `DYDX_TESTNET_MNEMONIC`) test account and testnet market
> state, not mainnet.

## `TradingVenue`

dYdX exposes a single exchange kind: the account's perpetual margin bucket.

In [2]:
async def exchanges() -> list[dict[str, str]]:
  """dYdX's only exchange kind is the perpetual margin bucket addressed by
  subaccount.
  """
  return [{'id': 'perp', 'type': 'perp'}]


await exchanges()

[{'id': 'perp', 'type': 'perp'}]

## `PerpExchange` (`perp`)

The parent subaccount (`0`) is addressed as `perp`; a child subaccount `N` would be
`perp.N` (`N % 128 == 0`) -- not exercised here since this testnet account only uses the
parent subaccount.

In [3]:
async def markets() -> list[str]:
  """List every perpetual market ticker known to the indexer."""
  data = await client.indexer.data.get_markets()
  return list(data['markets'])


markets_available = await markets()
len(markets_available), markets_available[:10]

(220,
 ['BTC-USD',
  'ETH-USD',
  'LINK-USD',
  'MATIC-USD',
  'CRV-USD',
  'SOL-USD',
  'ADA-USD',
  'AVAX-USD',
  'FIL-USD',
  'LTC-USD'])

In [4]:
fee_tier_response = await client.chain.feetiers.user_fee_tier(address)
if fee_tier_response.tier is None:
  raise ApiError('dYdX fee tier response did not include a tier')
fee_tier = fee_tier_response.tier
fee_tier

PerpetualFeeTier(name='1', maker_fee_ppm=100, taker_fee_ppm=500)

In [5]:
FUNDING_INTERVAL = timedelta(hours=1)  # dYdX settles funding hourly, on the hour


async def perp_stats(tickers: list[str]) -> dict[str, PerpStats]:
  """Fetch pricing and funding stats for many markets from one indexer call.

  dYdX reports no separate mark price, so `mark` is always `None`.
  """
  data = await client.indexer.data.get_markets()
  all_markets = data['markets']
  now = datetime.now().astimezone()
  next_time = now.replace(minute=0, second=0, microsecond=0) + FUNDING_INTERVAL
  out: dict[str, PerpStats] = {}
  for ticker in tickers:
    m = all_markets[ticker]
    open_interest = m.get('openInterest')
    index_price = m.get('oraclePrice')
    if index_price is None:
      raise ApiError(f'Oracle price unavailable for {ticker}')
    out[ticker] = PerpStats(
      index=Decimal(index_price),
      funding=Decimal(m['nextFundingRate']),
      next_funding_time=next_time,
      funding_interval=FUNDING_INTERVAL,
      open_interest=Decimal(open_interest) if open_interest is not None else None,
    )
  return out


await perp_stats(MARKETS)

{'BTC-USD': PerpStats(index=Decimal('79599.25992'), mark=None, funding=Decimal('0'), next_funding_time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), funding_interval=datetime.timedelta(seconds=3600), open_interest=Decimal('58.4404')),
 'ETH-USD': PerpStats(index=Decimal('2479.312931'), mark=None, funding=Decimal('0'), next_funding_time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), funding_interval=datetime.timedelta(seconds=3600), open_interest=Decimal('1095.517')),
 'SOL-USD': PerpStats(index=Decimal('105.48054805'), mark=None, funding=Decimal('0'), next_funding_time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), funding_interval=datetime.timedelta(seconds=3600), open_interest=Decimal('3394.18'))}

In [6]:
def parse_book(raw: OrderBook) -> Book:
  """Convert an indexer order book payload into an SDK `Book`."""
  return Book(
    asks=[
      Book.Entry(price=Decimal(level['price']), qty=Decimal(level['size']))
      for level in raw['asks']
    ],
    bids=[
      Book.Entry(price=Decimal(level['price']), qty=Decimal(level['size']))
      for level in raw['bids']
    ],
  )


async def tickers(tickers: list[str]) -> dict[str, Ticker]:
  """Fetch a ticker snapshot for many markets, enriched with top-of-book from each
  order book.
  """
  data = await client.indexer.data.get_markets()
  all_markets = data['markets']
  books = await asyncio.gather(
    *[client.indexer.data.get_order_book(t) for t in tickers]
  )
  out: dict[str, Ticker] = {}
  for ticker, raw_book in zip(tickers, books):
    m = all_markets[ticker]
    book = parse_book(raw_book)
    bid = book.bids[0] if book.bids else None
    ask = book.asks[0] if book.asks else None
    index_price = m.get('oraclePrice')
    out[ticker] = Ticker(
      last=Decimal(index_price) if index_price is not None else None,
      bid=bid.price if bid else None,
      ask=ask.price if ask else None,
      bid_qty=bid.qty if bid else None,
      ask_qty=ask.qty if ask else None,
      base_volume_24h=Decimal(m['volume24H']),
    )
  return out


await tickers(MARKETS)

{'BTC-USD': Ticker(last=Decimal('79599.25992'), bid=Decimal('79707'), ask=Decimal('80068'), bid_qty=Decimal('0.0001'), ask_qty=Decimal('0.0001'), base_volume_24h=Decimal('2770.4599')),
 'ETH-USD': Ticker(last=Decimal('2479.312931'), bid=Decimal('2485.1'), ask=Decimal('2491.4'), bid_qty=Decimal('0.001'), ask_qty=Decimal('0.001'), base_volume_24h=Decimal('203.5327')),
 'SOL-USD': Ticker(last=Decimal('105.48054805'), bid=Decimal('102.64'), ask=Decimal('109'), bid_qty=Decimal('9.74'), ask_qty=Decimal('9.17'), base_volume_24h=Decimal('8563.0760'))}

In [7]:
def effective_imf(market: PerpetualMarket) -> Decimal:
  """Compute the open-interest-scaled Initial Margin Fraction of a market.

  References:
    - [dYdX margin docs](https://docs.dydx.xyz/concepts/trading/margin#margining)
  """
  index_price = market.get('oraclePrice')
  if index_price is None:
    raise ApiError(f'Oracle price unavailable for {market["ticker"]}')
  open_notional = market['openInterest'] * Decimal(index_price)
  lower = market.get('openInterestLowerCap')
  upper = market.get('openInterestUpperCap')
  base_imf = market['initialMarginFraction']
  if lower is None or upper is None or upper == lower:
    return base_imf
  scale = (open_notional - lower) / (upper - lower)
  increase = scale * (1 - base_imf)
  return min(base_imf + max(increase, Decimal(0)), Decimal(1))


def effective_mmf(market: PerpetualMarket) -> Decimal:
  """Compute the open-interest-scaled Maintenance Margin Fraction of a market.

  Assumes the same OI-scaling factor applies to maintenance margin as to initial
  margin -- not independently confirmed against dYdX's docs, only that scaling *up* is
  the risk-safe direction (maintenance is over- rather than under-stated).
  """
  base_imf = market['initialMarginFraction']
  base_mmf = market['maintenanceMarginFraction']
  if base_imf == 0:
    return base_mmf
  return effective_imf(market) * base_mmf / base_imf


def max_leverage(market: PerpetualMarket) -> Decimal:
  """Return the maximum leverage implied by a market's margin metadata."""
  return Decimal(1) / effective_imf(market)


async def perp_collateral() -> PerpCollateral:
  """Fetch the exchange-level (parent subaccount `0`) perpetual collateral bucket."""
  sub, data = await asyncio.gather(
    client.indexer.data.get_subaccount(address, subaccount=0),
    client.indexer.data.get_markets(),
  )
  account = sub['subaccount']
  equity = Decimal(account['equity'])
  free_collateral = Decimal(account['freeCollateral'])
  all_markets = data['markets']
  notional = Decimal(0)
  maintenance_margin = Decimal(0)
  for position in account['openPerpetualPositions'].values():
    m = all_markets[position['market']]
    index_price = m.get('oraclePrice')
    if index_price is None:
      raise ApiError(f'Oracle price unavailable for {position["market"]}')
    position_notional = abs(Decimal(position['size'])) * Decimal(index_price)
    notional += position_notional
    maintenance_margin += position_notional * effective_mmf(m)
  leverage = notional / equity if equity > 0 else Decimal(0)
  return PerpCollateral(
    equity=equity,
    free_collateral=free_collateral,
    initial_margin=equity - free_collateral,
    maintenance_margin=maintenance_margin,
    leverage=leverage,
    margin_mode='cross',  # parent subaccounts (< 128) are always the cross-margin pool
  )


await perp_collateral()

PerpCollateral(equity=Decimal('10079.015190528'), free_collateral=Decimal('10069.62247785744'), initial_margin=Decimal('9.39271267056'), maintenance_margin=Decimal('5.635627602336'), leverage=Decimal('0.04659538899885293600878139462'), margin_mode='cross')

## `PerpMarket`

dYdX's only exchange kind is `perp` (no spot market), so this section covers the whole
`Market`/`PerpMarket` interface directly, for `BTC-USD`, `ETH-USD`, and `SOL-USD`.

In [8]:
all_markets = (await client.indexer.data.get_markets())['markets']
market_info: dict[str, PerpetualMarket] = {t: all_markets[t] for t in MARKETS}

In [9]:
async def depth(ticker: str) -> Book:
  raw = await client.indexer.data.get_order_book(ticker)
  return parse_book(raw)


{t: await depth(t) for t in MARKETS}

{'BTC-USD': Book(bids=[Book.Entry(price=Decimal('79707'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('79627'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('77490'), qty=Decimal('0.0129')), Book.Entry(price=Decimal('4000'), qty=Decimal('0.1'))], asks=[Book.Entry(price=Decimal('80068'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('80107'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('80148'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('82285'), qty=Decimal('0.0121'))]),
 'ETH-USD': Book(bids=[Book.Entry(price=Decimal('2413.6'), qty=Decimal('0.828'))], asks=[]),
 'SOL-USD': Book(bids=[Book.Entry(price=Decimal('102.64'), qty=Decimal('9.74'))], asks=[Book.Entry(price=Decimal('109'), qty=Decimal('9.17'))])}

In [ ]:
def apply_book_update(book: Book, update: OrderbookMessageContents) -> None:
  """Apply an incremental order book message onto a local `Book`, in place."""
  book.update(
    Book(
      asks=[Book.Entry(price, qty) for price, qty in update.get('asks', [])],
      bids=[Book.Entry(price, qty) for price, qty in update.get('bids', [])],
    )
  )


async def depth_stream(
  ticker: str, *, count: int = 3, timeout: float = 15.0
) -> list[Book]:
  """Subscribe to the order book and collect up to `count` updated snapshots."""
  raise NotImplementedError(
    'blocked: typed_dydx validates the v4_orderbook subscription reply against the '
    'notification type (OrderbookMessageContents), so entering the stream raises '
    'ValidationError before the first message'
  )
  books: list[Book] = []
  async with client.indexer.streams.orders(id=ticker) as stream:
    book = parse_book(stream.reply)
    it = aiter(stream)
    try:
      while len(books) < count:
        msg = await asyncio.wait_for(anext(it), timeout=timeout)
        apply_book_update(book, msg)
        books.append(book.copy())
    except asyncio.TimeoutError:
      pass
  return books


# not executed: blocked by typed-dydx "Stream subscription replies are validated with the channel's notification type"
await depth_stream('BTC-USD')

In [10]:
def fee_ppm(value: int) -> Decimal:
  """Convert dYdX fee parts-per-million into a decimal rate."""
  return Decimal(value) / Decimal(1_000_000)


async def rules(ticker: str) -> Rules:
  m = market_info[ticker]
  base, quote = m['ticker'].split('-')
  return Rules(
    base=base,
    quote=quote,
    fee_asset=quote,
    tick_size=Decimal(m['tickSize']),
    step_size=Decimal(m['stepSize']),
    maker_fee=fee_ppm(fee_tier.maker_fee_ppm),
    taker_fee=fee_ppm(fee_tier.taker_fee_ppm),
    api=m['status'] == 'ACTIVE',
    details={'perpetual_market': m, 'user_fees': fee_tier},
  )


{t: await rules(t) for t in MARKETS}

{'BTC-USD': Rules(base='BTC', quote='USD', fee_asset='USD', tick_size=Decimal('1'), step_size=Decimal('0.0001'), fixed_min_qty=None, min_value=None, max_qty=None, fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.0001'), taker_fee=Decimal('0.0005'), api=True, details={'perpetual_market': {'atomicResolution': -10, 'baseOpenInterest': Decimal('50.3776'), 'clobPairId': 0, 'defaultFundingRate1H': Decimal('0'), 'initialMarginFraction': Decimal('0.02'), 'maintenanceMarginFraction': Decimal('0.012'), 'marketType': 'CROSS', 'nextFundingRate': Decimal('0'), 'openInterest': Decimal('58.4404'), 'openInterestLowerCap': Decimal('0'), 'openInterestUpperCap': Decimal('0'), 'oraclePrice': Decimal('79599.25992'), 'priceChange24H': Decimal('-198.49913'), 'quantumConversionExponent': -9, 'status': 'ACTIVE', 'stepBaseQuantums': 1000000, 'stepSize': Decimal('0.0001'), 'subticksPerTick': 100000, 'tickSize': Decimal('1'), 'ticker': 'BTC-USD', 'trades24H'

In [11]:
import base64
from typed_dydx.protos.dydxprotocol import clob, subaccounts


def order_active(status: str) -> bool:
  return status in {'OPEN', 'PENDING', 'UNTRIGGERED', 'BEST_EFFORT_OPENED'}


def order_sign(side: str) -> int:
  return 1 if side == 'BUY' else -1


def protobuf_order_id(order: IndexerOrder) -> clob.OrderId:
  """Build a protocol order ID for an indexer order."""
  subaccount_number = order.get('subaccountNumber')
  if subaccount_number is None:
    raise ValidationError('dYdX order did not include a subaccount number')
  return clob.OrderId(
    client_id=int(order['clientId']),
    order_flags=int(order['orderFlags']),
    clob_pair_id=int(order['clobPairId']),
    subaccount_id=subaccounts.SubaccountId(
      owner=address, number=int(subaccount_number)
    ),
  )


def serialize_order_id(order_id: clob.OrderId) -> str:
  """Serialize a dYdX protocol order ID for the SDK order API."""
  return base64.b64encode(bytes(order_id)).decode()


def parse_order_id(id: str) -> clob.OrderId:
  """Parse an SDK order ID back into a dYdX protocol order ID."""
  return clob.OrderId.FromString(base64.b64decode(id))


def parse_order_state(order: IndexerOrder) -> OrderState:
  sign = order_sign(order['side'])
  return OrderState(
    id=serialize_order_id(protobuf_order_id(order)),
    price=Decimal(order['price']),
    qty=Decimal(order['size']) * sign,
    filled_qty=Decimal(order['totalFilled']) * sign,
    active=order_active(order['status']),
    details=order,
  )


async def list_orders(
  ticker: str, *, status: OrderStatus | None = None
) -> list[OrderState]:
  orders = await client.indexer.data.list_parent_orders(
    address=address,
    parent_subaccount=0,
    ticker=ticker,
    status=status,
  )
  return [parse_order_state(o) for o in orders]


async def open_orders(ticker: str) -> list[OrderState]:
  return await list_orders(ticker, status='OPEN')


async def query_order(ticker: str, id: str) -> OrderState | None:
  for order in await list_orders(ticker):
    if order.id == id:
      return order


{t: await open_orders(t) for t in MARKETS}

{'BTC-USD': [], 'ETH-USD': [], 'SOL-USD': []}

In [12]:
async def trades_history(ticker: str, start: datetime, end: datetime) -> list[Trade]:
  start = start.astimezone()
  end = end.astimezone()
  pages = client.indexer.data.get_fills_paged(
    address=address,
    subaccount=0,
    created_before_or_at=end,
    market=ticker,
    market_type='PERPETUAL',
  )
  out: list[Trade] = []
  async for page in pages:
    for f in page:
      if not (start <= f['createdAt'] <= end):
        continue
      sign = order_sign(f['side'])
      out.append(
        Trade(
          id=f['id'],
          price=Decimal(f['price']),
          qty=Decimal(f['size']) * sign,
          time=f['createdAt'],
          maker=f['liquidity'] == 'MAKER',
          fee=Trade.Fee(asset='USDC', amount=Decimal(f['fee'])),
          details=f,
        )
      )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(days=90)
{t: await trades_history(t, start, end) for t in MARKETS}

{'BTC-USD': [Trade(id='ff627473-9aea-521f-a6f4-8306bb9bd4c9', price=Decimal('59907'), qty=Decimal('-0.0015'), time=datetime.datetime(2026, 7, 6, 7, 35, 43, 165000, tzinfo=datetime.timezone.utc), maker=False, fee=Trade.Fee(amount=Decimal('0.044931'), asset='USDC'), details={'affiliateRevShare': Decimal('0'), 'clientMetadata': '1', 'createdAt': datetime.datetime(2026, 7, 6, 7, 35, 43, 165000, tzinfo=datetime.timezone.utc), 'createdAtHeight': 79216857, 'entryPriceBefore': Decimal('53622.66666666666666666667'), 'fee': Decimal('0.044931'), 'id': 'ff627473-9aea-521f-a6f4-8306bb9bd4c9', 'liquidity': 'TAKER', 'market': 'BTC-USD', 'marketType': 'PERPETUAL', 'orderId': '01bb1caa-3df1-5d9a-8726-3c5d9a5a9d64', 'positionSideBefore': 'LONG', 'positionSizeBefore': Decimal('0.0015'), 'price': Decimal('59907'), 'side': 'SELL', 'size': Decimal('0.0015'), 'subaccountNumber': 0, 'type': 'LIMIT'}),
  Trade(id='0aefe850-4f8f-5149-a6f7-3bcc62b1f194', price=Decimal('63164'), qty=Decimal('0.001'), time=datetim

In [13]:
async def trades_stream_sample(ticker: str, *, timeout: float = 5.0):
  """Subscribe to real-time fills and return the first one matching `ticker`, or a
  timeout marker if none arrive within `timeout` seconds.
  """
  async with client.indexer.streams.parent_subaccounts(address, subaccount=0) as stream:
    it = aiter(stream)
    try:
      msg = await asyncio.wait_for(anext(it), timeout=timeout)
    except asyncio.TimeoutError:
      return (
        'no new trades observed in 5s (expected -- no live trading on this account)'
      )
    for fill in msg.get('fills') or []:
      if fill['ticker'] != ticker:
        continue
      sign = order_sign(fill['side'])
      return Trade(
        id=fill['id'],
        price=Decimal(fill['price']),
        qty=Decimal(fill['size']) * sign,
        time=fill['createdAt'],
        maker=fill['liquidity'] == 'MAKER',
        fee=None,
        details=fill,
      )
    return msg  # no matching fill in this notification -- return it raw for inspection


await trades_stream_sample('BTC-USD')

'no new trades observed in 5s (expected -- no live trading on this account)'

In [14]:
await query_order('BTC-USD', 'nonexistent-order-id')

In [15]:
async def perp_position(ticker: str) -> PerpPosition:
  positions = await client.indexer.data.list_parent_positions(
    address, parent_subaccount=0
  )
  matches = [p for p in positions if p['market'] == ticker and p['status'] == 'OPEN']
  if not matches:
    return PerpPosition()
  total_size = sum((Decimal(p['size']) for p in matches), Decimal(0))
  if total_size == 0:
    return PerpPosition()
  total_notional = sum(
    (Decimal(p['size']) * Decimal(p['entryPrice']) for p in matches), Decimal(0)
  )
  return PerpPosition(size=total_size, entry_price=total_notional / total_size)


{t: await perp_position(t) for t in MARKETS}

{'BTC-USD': PerpPosition(size=Decimal('0.0059'), entry_price=Decimal('64882.33898305084745762712')),
 'ETH-USD': PerpPosition(size=Decimal('0'), entry_price=Decimal('0')),
 'SOL-USD': PerpPosition(size=Decimal('0'), entry_price=Decimal('0'))}

In [16]:
async def market_perp_collateral(ticker: str) -> PerpCollateral:
  """dYdX has no per-market margin mode: a market's collateral bucket is exactly its
  exchange's parent-subaccount pool.
  """
  return await perp_collateral()


{t: await market_perp_collateral(t) for t in MARKETS}

{'BTC-USD': PerpCollateral(equity=Decimal('10079.015190528'), free_collateral=Decimal('10069.62247785744'), initial_margin=Decimal('9.39271267056'), maintenance_margin=Decimal('5.635627602336'), leverage=Decimal('0.04659538899885293600878139462'), margin_mode='cross'),
 'ETH-USD': PerpCollateral(equity=Decimal('10079.015190528'), free_collateral=Decimal('10069.62247785744'), initial_margin=Decimal('9.39271267056'), maintenance_margin=Decimal('5.635627602336'), leverage=Decimal('0.04659538899885293600878139462'), margin_mode='cross'),
 'SOL-USD': PerpCollateral(equity=Decimal('10079.015190528'), free_collateral=Decimal('10069.62247785744'), initial_margin=Decimal('9.39271267056'), maintenance_margin=Decimal('5.635627602336'), leverage=Decimal('0.04659538899885293600878139462'), margin_mode='cross')}

In [17]:
async def available_notional(ticker: str) -> Decimal:
  sub, m = await asyncio.gather(
    client.indexer.data.get_subaccount(address, subaccount=0),
    client.indexer.data.get_market(ticker),
  )
  free_collateral = Decimal(sub['subaccount']['freeCollateral'])
  return free_collateral * max_leverage(m)


{t: await available_notional(t) for t in MARKETS}

{'BTC-USD': Decimal('503481.1238928720'),
 'ETH-USD': Decimal('503481.1238928720'),
 'SOL-USD': Decimal('201392.4495571488')}

In [18]:
async def index(ticker: str) -> Decimal:
  m = await client.indexer.data.get_market(ticker)
  index_price = m.get('oraclePrice')
  if index_price is None:
    raise ApiError(f'Oracle price unavailable for {ticker}')
  return Decimal(index_price)


{t: await index(t) for t in MARKETS}

{'BTC-USD': Decimal('79599.25992'),
 'ETH-USD': Decimal('2479.312931'),
 'SOL-USD': Decimal('105.48054805')}

In [19]:
async def next_funding(ticker: str) -> NextFunding:
  m = await client.indexer.data.get_market(ticker)
  now = datetime.now().astimezone()
  next_time = now.replace(minute=0, second=0, microsecond=0) + FUNDING_INTERVAL
  return NextFunding(
    rate=Decimal(m['nextFundingRate']), time=next_time, interval=FUNDING_INTERVAL
  )


{t: await next_funding(t) for t in MARKETS}

{'BTC-USD': NextFunding(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), premium=None, interval=datetime.timedelta(seconds=3600)),
 'ETH-USD': NextFunding(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), premium=None, interval=datetime.timedelta(seconds=3600)),
 'SOL-USD': NextFunding(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), premium=None, interval=datetime.timedelta(seconds=3600))}

In [20]:
async def funding_rates(
  ticker: str, start: datetime | None, end: datetime | None
) -> list[FundingRate]:
  start = start.astimezone() if start is not None else None
  end = end.astimezone() if end is not None else None
  pages = client.indexer.data.get_historical_funding_paged(
    ticker, effective_before_or_at=end
  )
  out: list[FundingRate] = []
  async for page in pages:
    for item in page:
      if start is None or item['effectiveAt'] >= start:
        out.append(FundingRate(rate=Decimal(item['rate']), time=item['effectiveAt']))
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(days=2)
{t: await funding_rates(t, start, end) for t in MARKETS}

Task was destroyed but it is pending!
task: <Task pending name='Task-67' coro=<Queue.get() running at /home/ubuntu/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/asyncio/queues.py:158> wait_for=<Future pending cb=[Task.task_wakeup()]>>


{'BTC-USD': [FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 15, 0, 0, 657000, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 14, 0, 0, 674000, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 13, 0, 0, 402000, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 12, 0, 0, 449000, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 11, 0, 0, 272000, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 10, 0, 0, 448000, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6, 9, 0, 0, 688000, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0'), time=datetime.datetime(2026, 9, 6

In [21]:
async def funding_payments(
  ticker: str, start: datetime, end: datetime
) -> list[FundingPayment]:
  start = start.astimezone()
  end = end.astimezone()
  pages = client.indexer.data.get_funding_payments_paged(
    address=address,
    subaccount=0,
    ticker=ticker,
    after_or_at=start,
  )
  out: list[FundingPayment] = []
  async for page in pages:
    for item in page:
      if start <= item['createdAt'] <= end:
        out.append(
          FundingPayment(amount=Decimal(item['payment']), time=item['createdAt'])
        )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(days=90)
{t: await funding_payments(t, start, end) for t in MARKETS}

{'BTC-USD': [FundingPayment(amount=Decimal('-0.122956'), time=datetime.datetime(2026, 9, 1, 15, 0, 0, 866000, tzinfo=datetime.timezone.utc)),
  FundingPayment(amount=Decimal('-2.761141'), time=datetime.datetime(2026, 9, 1, 13, 0, 0, 567000, tzinfo=datetime.timezone.utc)),
  FundingPayment(amount=Decimal('-0.020355'), time=datetime.datetime(2026, 9, 1, 9, 0, 0, 984000, tzinfo=datetime.timezone.utc)),
  FundingPayment(amount=Decimal('-2.787809'), time=datetime.datetime(2026, 9, 1, 7, 0, 0, 18000, tzinfo=datetime.timezone.utc)),
  FundingPayment(amount=Decimal('0.106259'), time=datetime.datetime(2026, 9, 1, 6, 0, 0, 572000, tzinfo=datetime.timezone.utc)),
  FundingPayment(amount=Decimal('0.041610'), time=datetime.datetime(2026, 6, 15, 11, 0, 0, 685000, tzinfo=datetime.timezone.utc)),
  FundingPayment(amount=Decimal('0.030075'), time=datetime.datetime(2026, 6, 15, 10, 0, 0, 393000, tzinfo=datetime.timezone.utc)),
  FundingPayment(amount=Decimal('0.041370'), time=datetime.datetime(2026, 6, 

In [ ]:
from typed_dydx.node.orders.types import ShortTermOrderParams


async def place_order(ticker: str, order: Order) -> OrderResponse:
  signed_qty = Decimal(order['qty'])
  side = 'BUY' if signed_qty >= 0 else 'SELL'
  params = ShortTermOrderParams(
    side=side,
    price=Decimal(order['price']),
    size=abs(signed_qty),
    time_in_force='IMMEDIATE_OR_CANCEL'
    if order['type'] == 'MARKET'
    else 'GOOD_TIL_TIME',
    flags='SHORT_TERM',
  )
  response = await client.node.place_order(
    market_info[ticker], order=params, subaccount=0
  )
  order_id = response.order.order_id
  if order_id is None:
    raise ValidationError('dYdX place order response did not include an order ID')
  return OrderResponse(id=serialize_order_id(order_id), details=response)


order: Order = {'qty': Decimal('0.001'), 'price': Decimal('20000'), 'type': 'LIMIT'}
# Not executed here -- would place a real order on the testnet account.
await place_order('BTC-USD', order)

In [ ]:
async def cancel_order(id: str):
  return await client.node.cancel_order(parse_order_id(id))


# Not executed here -- would cancel a real order on the testnet account.
await cancel_order('some-order-id')

In [ ]:
async def cancel_orders(ids: list[str]):
  order_ids = [parse_order_id(id) for id in ids]
  return await client.node.batch_cancel_orders(order_ids)


# Not executed here -- would cancel real orders on the testnet account.
await cancel_orders(['id-1', 'id-2'])

## Coverage assessment

**Full bar one cell**, executed live against `BTC-USD`, `ETH-USD`, and `SOL-USD` on the
testnet account -- every cell above hand-maps a raw `typed_dydx` response onto a
`tribulnation.sdk.market` type directly, with no `tribulnation.dydx` import at all:

- `TradingVenue`: `exchanges()` ran live and returns dYdX's single exchange kind
  (`perp`). dYdX has no spot market, so there is no second, non-perp `TradingVenue`
  accessor to exercise separately.
- `PerpExchange` (`perp`): `markets()` (220+ real perpetuals), `perp_stats()`,
  `tickers()`, and the exchange-level `perp_collateral()` (parent subaccount `0`, the
  account's only subaccount on this testnet wallet) all ran live and returned real data.
- `PerpMarket`: every method except `depth_stream` ran live and returned real,
  non-fabricated data (or a real empty/`None` result) -- `depth` returned real testnet
  order books; `rules` returned real tick/step size and fee-tier data; `open_orders` came
  back empty (no resting orders); `trades_history` found real historical fills on
  `BTC-USD` over the last 90 days and none on `ETH-USD`/`SOL-USD`; `trades_stream_sample`
  correctly reported
  no live fill within a 5s window (no active trading during the run); `query_order`
  correctly returned `None` for a nonexistent id; `perp_position` shows a real open
  `BTC-USD` long and flat `ETH-USD`/`SOL-USD`; `perp_collateral`/`available_notional`
  reflect that same real account state; `index`/`next_funding` read the indexer's live
  oracle price and `nextFundingRate` field (dYdX funds hourly, so `next_funding`'s `time`
  is computed as the top of the next hour rather than read -- there is no separate
  "next funding timestamp" field to read); `funding_rates`/`funding_payments` both
  returned real historical entries.
- `depth_stream` is **blocked** on typed-dydx "Stream subscription replies are
  validated with the channel's notification type": `StreamsMixin.subscribe` validates
  the `v4_orderbook` reply (`{"price", "size"}` objects, `OrderbookReplyContents`)
  against the notification type (`[price, size]` tuples, `OrderbookMessageContents`),
  so entering the stream raises `ValidationError` before the first message. The mapping
  is written; its body raises `NotImplementedError` and the cell is not executed until
  the client takes a `reply_type`.
- `Market.position()`/`Market.collateral()` (the base, non-perp-named methods) are not
  exercised as separate cells -- `PerpMarket` implements them as trivial delegators to
  `perp_position()`/`perp_collateral()`, both already exercised above; this notebook's
  hand-written functions mirror that delegation with `market_perp_collateral`.
- `place_order`, `cancel_order`, and `cancel_orders` are written but **not executed** --
  they would place/cancel a real order on the testnet account. `place_orders`,
  `cancel_open_orders` (the SDK's `asyncio.gather`-over-single-method defaults) are not
  separately hand-mapped since they add no venue-specific logic beyond what's above.